# ParkCast SF — Data Preprocessing

Aggregates raw transaction logs into hourly occupancy percentages (%),
enriches the data with weather, street sweeping, event, and citation features.

**Output:** `data/processed_training_data.csv`

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## Config

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())  # project root (run notebook from dev/)
DATA_DIR = os.path.join(PROJECT_DIR, "data")
OUTPUT_PATH = os.path.join(DATA_DIR, "processed_training_data.csv")

# 2026 Holidays (Partial list for labelling)
HOLIDAYS_2026 = [
    "2026-01-01", "2026-01-19", "2026-02-16", "2026-04-05", # Added Easter Sunday
    "2026-05-25", "2026-06-19", "2026-07-04", "2026-09-07", 
    "2026-10-12", "2026-11-11", "2026-11-26", "2026-12-25"
]

## `is_school_day()`

Simple heuristic for SF school days (Mon-Fri, excluding summer/winter).

In [3]:
def is_school_day(dt):
    """Simple heuristic for SF school days (Mon-Fri, excluding summer/winter)."""
    if dt.weekday() >= 5: return 0
    # SFUSD Spring Break 2026: March 27 - April 3
    date_only = dt.date()
    if date_only >= datetime(2026, 3, 27).date() and date_only <= datetime(2026, 4, 3).date():
        return 0
        
    return 1

## `load_data()`

In [4]:
def load_data():
    print("Loading raw datasets...")
    df_trans = pd.read_csv(os.path.join(DATA_DIR, "meter_transactions.csv"))
    df_locs = pd.read_csv(os.path.join(DATA_DIR, "meter_locations.csv"))
    df_weather = pd.read_csv(os.path.join(DATA_DIR, "weather.csv"))
    df_sweeping = pd.read_csv(os.path.join(DATA_DIR, "street_sweeping.csv"))
    
    # Optional datasets
    try:
        df_closures = pd.read_csv(os.path.join(DATA_DIR, "street_closures.csv"))
    except:
        df_closures = None
        
    return df_trans, df_locs, df_weather, df_sweeping, df_closures

## `calculate_occupancy()`

In [5]:
def calculate_occupancy(df_trans, df_locs):
    """
    Build a full (active_blockface × hour) grid. For each cell, compute
    occupancy as TOTAL PAID MINUTES / (n_meters × 60), not the old
    "count of meters with ≥1 session in the hour". The old count treated
    a meter with a 5-minute session and a meter parked solid for the full
    hour identically — this version captures intensity, not just presence.

    Numerator is summed per-meter-hour (capped at 60 min) so any
    pathological overlapping sessions on a single post can't double-count.
    """
    print("Calculating hourly occupancy from transactions (minutes-paid)...")

    loc_cols = ['post_id', 'blockface_id', 'analysis_neighborhood', 'latitude', 'longitude']
    df_locs = df_locs.dropna(subset=['blockface_id', 'post_id'])[loc_cols]

    df = df_trans.merge(df_locs, on='post_id', how='inner').reset_index(drop=True)
    df['start'] = pd.to_datetime(df['session_start_dt'])
    df['end'] = pd.to_datetime(df['session_end_dt'])

    # Drop degenerate rows: negative or absurdly long sessions
    dur_hrs = (df['end'] - df['start']).dt.total_seconds() / 3600
    df = df[(dur_hrs > 0) & (dur_hrs <= 24)].reset_index(drop=True)

    # Hour grid spans the whole transaction window
    start_hour = df['start'].min().floor('h')
    end_hour = df['end'].max().floor('h')
    hours = pd.date_range(start=start_hour, end=end_hour, freq='h')

    # Blockface capacity: only count meter posts that actually generate
    # transactions. Dead meters inflate the denominator.
    active_posts = set(df['post_id'].dropna().unique())
    df_locs_active = df_locs[df_locs['post_id'].isin(active_posts)]
    block_capacity = df_locs_active.groupby('blockface_id')['post_id'].nunique()
    block_meta = df_locs_active.groupby('blockface_id').agg(
        neighborhood=('analysis_neighborhood', 'first'),
        lat=('latitude', 'mean'),
        lon=('longitude', 'mean'),
    )

    MIN_ACTIVE_POSTS = 4
    valid_bfs = block_capacity[block_capacity >= MIN_ACTIVE_POSTS].index
    block_capacity = block_capacity.loc[valid_bfs]
    block_meta = block_meta.loc[valid_bfs]
    df = df[df['blockface_id'].isin(valid_bfs)].reset_index(drop=True)
    print(f"  Blockfaces with >= {MIN_ACTIVE_POSTS} active meters: {len(valid_bfs):,}")

    active_blocks = df['blockface_id'].dropna().unique()
    print(f"  Active blockfaces: {len(active_blocks)} | Hours: {len(hours)} "
          f"| Grid: {len(active_blocks) * len(hours):,} rows")

    # Expand each session into the hourly bins it touches, then compute
    # the actual overlap in minutes with each bin.
    df['start_h'] = df['start'].dt.floor('h')
    df['n_hours'] = ((df['end'].dt.ceil('h') - df['start_h']).dt.total_seconds() / 3600).astype(int)
    df = df[df['n_hours'] > 0].reset_index(drop=True)

    rep = df.loc[df.index.repeat(df['n_hours']),
                 ['blockface_id', 'post_id', 'start_h', 'start', 'end']].copy()
    rep['offset'] = rep.groupby(rep.index).cumcount()
    rep['hour'] = rep['start_h'] + pd.to_timedelta(rep['offset'], unit='h')
    rep['hour_end'] = rep['hour'] + pd.Timedelta(hours=1)

    # Vectorized minutes-of-overlap = min(end, hour+1h) - max(start, hour)
    s = rep['start'].values.astype('datetime64[ns]')
    e = rep['end'].values.astype('datetime64[ns]')
    hs = rep['hour'].values.astype('datetime64[ns]')
    he = rep['hour_end'].values.astype('datetime64[ns]')
    overlap_ns = np.minimum(e, he) - np.maximum(s, hs)
    rep['minutes'] = (overlap_ns / np.timedelta64(1, 'm')).clip(0, 60)

    # Per-meter-hour minutes (cap 60 guards against overlapping sessions
    # on the same post in the same hour — shouldn't happen, belt and braces)
    meter_hour = (rep.groupby(['blockface_id', 'post_id', 'hour'])['minutes']
                     .sum()
                     .clip(upper=60)
                     .reset_index())

    # Block-hour: total paid minutes across all meters on the block
    occ = (meter_hour.groupby(['blockface_id', 'hour'])['minutes']
                     .sum()
                     .reset_index(name='paid_minutes'))

    # Full cross-product grid, left-joined to paid minutes (missing = 0)
    grid = (pd.MultiIndex
              .from_product([active_blocks, hours], names=['blockface_id', 'timestamp'])
              .to_frame(index=False))
    grid = grid.merge(occ, left_on=['blockface_id', 'timestamp'],
                      right_on=['blockface_id', 'hour'], how='left')
    grid['paid_minutes'] = grid['paid_minutes'].fillna(0.0)
    grid = grid.drop(columns=['hour'])

    grid = grid.merge(block_meta, on='blockface_id', how='left')
    grid['total_spaces'] = grid['blockface_id'].map(block_capacity).astype(int)
    # occupancy_pct = total_paid_minutes / (n_meters × 60) × 100
    grid['occupancy_pct'] = (grid['paid_minutes']
                             / (grid['total_spaces'] * 60.0)
                             * 100).clip(upper=100).round(2)

    return grid[['timestamp', 'blockface_id', 'neighborhood',
                 'lat', 'lon', 'occupancy_pct', 'total_spaces']]

## `enrich_features()`

In [6]:
def enrich_features(df, df_weather, df_sweeping, df_closures):
    print("Enriching with features (Weather, School, Holiday, Events)...")
    
    # Time features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.weekday
    df['month'] = df['timestamp'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Labels
    df['is_holiday'] = df['timestamp'].dt.strftime('%Y-%m-%d').isin(HOLIDAYS_2026).astype(int)
    df['is_school_day'] = df['timestamp'].apply(is_school_day)
    
    # Weather
    print("  Merging weather...")
    df_weather['time'] = pd.to_datetime(df_weather['time'])
    df = df.merge(df_weather[['time', 'temperature_2m', 'precipitation']], left_on='timestamp', right_on='time', how='left')
    df.rename(columns={'temperature_2m': 'temperature', 'precipitation': 'is_raining'}, inplace=True)
    df['is_raining'] = (df['is_raining'] > 0).astype(int)
    
    # Street Sweeping (Simplified check: if neighborhood matches and day matches)
    # Real SF sweeping is complex; we'll add a placeholder binary feature based on neighborhood patterns
    df['is_street_cleaning'] = 0 # would be calculated by joining df_sweeping CNNs
    
    # Events — continuous intensity = max over active events of
    # exp(-dist/300) within 800m, during event hours. Captures both
    # distance falloff (~0.07 at 800m, 1.0 at venue) and presence/absence.
    df['event_intensity'] = 0.0
    events_path = os.path.join(DATA_DIR, "events.csv")
    if os.path.exists(events_path):
        print("  Merging events (continuous intensity)...")
        df_ev = pd.read_csv(events_path)
        df_ev['date'] = pd.to_datetime(df_ev['date']).dt.date
        df['date_only'] = df['timestamp'].dt.date

        R = 6_371_000
        lat_rad = np.radians(df['lat'].values)
        lon_rad = np.radians(df['lon'].values)
        intensity = np.zeros(len(df), dtype=np.float32)

        for _, ev in df_ev.iterrows():
            v_lat = np.radians(ev['venue_lat'])
            v_lon = np.radians(ev['venue_lon'])
            dlat = lat_rad - v_lat
            dlon = lon_rad - v_lon
            a = np.sin(dlat / 2) ** 2 + np.cos(lat_rad) * np.cos(v_lat) * np.sin(dlon / 2) ** 2
            dist = 2 * R * np.arcsin(np.sqrt(a))

            time_mask = (
                (df['date_only'].values == ev['date'])
                & (df['hour'].values >= ev['start_hour'])
                & (df['hour'].values <= ev['end_hour'])
            )
            close = time_mask & (dist <= 800)
            if close.any():
                contrib = np.exp(-dist[close] / 300.0).astype(np.float32)
                idx = np.where(close)[0]
                np.maximum.at(intensity, idx, contrib)

        df['event_intensity'] = intensity
        df.drop(columns=['date_only'], inplace=True)

    return df

## `merge_citations()`

In [7]:
def merge_citations(df):
    """
    Join hourly citation counts per blockface. Also compute a robust
    per-(block, hour-of-day, weekday) median so hours with zero citations
    still carry the block's typical signal for that slot.
    """
    cit_path = os.path.join(DATA_DIR, "citations_by_block.csv")
    if not os.path.exists(cit_path):
        print("  No citations_by_block.csv found — skipping citation features.")
        df['citation_count'] = 0.0
        df['citations_hourly_median'] = 0.0
        return df

    print("  Merging citation signal...")
    cit = pd.read_csv(cit_path, parse_dates=['timestamp'])

    # Raw per-hour count
    df = df.merge(cit, on=['blockface_id', 'timestamp'], how='left')
    df['citation_count'] = df['citation_count'].fillna(0.0)

    # Historical median citations per (blockface, hour-of-day, weekday)
    cit['hour_of_day'] = cit['timestamp'].dt.hour
    cit['weekday'] = cit['timestamp'].dt.weekday
    median = (cit.groupby(['blockface_id', 'hour_of_day', 'weekday'])
                 ['citation_count'].median()
                 .reset_index(name='citations_hourly_median'))
    df = df.merge(median,
                  left_on=['blockface_id', 'hour', 'day_of_week'],
                  right_on=['blockface_id', 'hour_of_day', 'weekday'],
                  how='left')
    df['citations_hourly_median'] = df['citations_hourly_median'].fillna(0.0)
    df = df.drop(columns=['hour_of_day', 'weekday'], errors='ignore')
    return df

## `merge_poi_features()` & `merge_311_features()`

POI density is a static per-block feature keyed on the master `cnn`; we map
each SFMTA blockface centroid to its nearest `cnn` once and carry the POI
counts through.

311 complaint rates are keyed directly on `blockface_id × hour × weekday`
(same shape as `citations_hourly_median`), so it joins straight onto the
training rows.

In [8]:
POI_COLS = ["poi_dining_200m", "poi_retail_200m",
            "poi_transit_200m", "poi_attraction_200m"]

def merge_poi_features(df):
    """Attach OSM POI density (dining / retail / transit / attraction) per
    block. Keyed on cnn in the feature file; we map each training (lat, lon)
    to its nearest master block to get a cnn."""
    poi_path = os.path.join(DATA_DIR, "block_poi_features.csv")
    master_path = os.path.join(PROJECT_DIR, "app", "models", "master_blocks.parquet")
    if not os.path.exists(poi_path) or not os.path.exists(master_path):
        print("  POI features or master_blocks missing — skipping POI merge.")
        for c in POI_COLS:
            df[c] = 0
        return df

    print("  Merging POI density features...")
    from scipy.spatial import cKDTree
    poi = pd.read_csv(poi_path)
    master = pd.read_parquet(master_path)[["cnn", "lat", "lon"]]

    # One lookup per unique (lat, lon) in the training data
    unique_pts = df[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
    tree = cKDTree(master[["lat", "lon"]].values)
    _, idx = tree.query(unique_pts[["lat", "lon"]].values, k=1)
    unique_pts["cnn"] = master.iloc[idx]["cnn"].values

    unique_pts = unique_pts.merge(poi, on="cnn", how="left")
    for c in POI_COLS:
        if c not in unique_pts.columns:
            unique_pts[c] = 0
        unique_pts[c] = unique_pts[c].fillna(0).astype(int)

    df = df.merge(unique_pts[["lat", "lon"] + POI_COLS],
                  on=["lat", "lon"], how="left")
    for c in POI_COLS:
        df[c] = df[c].fillna(0).astype(int)
    print(f"    POI coverage: "
          f"{(df[POI_COLS].sum(axis=1) > 0).mean() * 100:.1f}% of rows")
    return df


def merge_311_features(df):
    """Attach median daily 311-parking-complaint count per
    (blockface_id, hour, weekday). Zero where no signal exists."""
    path = os.path.join(DATA_DIR, "block_311_features.csv")
    if not os.path.exists(path):
        print("  block_311_features.csv missing — skipping 311 merge.")
        df["complaints_311_median"] = 0.0
        df["complaints_311_total"] = 0.0
        return df

    print("  Merging 311 complaint features...")
    f = pd.read_csv(path)
    df = df.merge(
        f,
        left_on=["blockface_id", "hour", "day_of_week"],
        right_on=["blockface_id", "hour_of_day", "weekday"],
        how="left",
    )
    df = df.drop(columns=["hour_of_day", "weekday"], errors="ignore")
    df["complaints_311_median"] = df["complaints_311_median"].fillna(0.0)
    df["complaints_311_total"] = df["complaints_311_total"].fillna(0.0)
    coverage = (df["complaints_311_median"] > 0).mean() * 100
    print(f"    311 coverage: {coverage:.1f}% of rows have nonzero median")
    return df

## `merge_sfpark_features()` & `merge_permits_features()`

**SFpark block occupancy** — physical occupancy from the 2011-2013 sensor
pilot, aggregated per `(blockface_id, hour, is_weekend)`. Available only on
pilot blocks (~8 districts). Serves two roles:

1. *Training feature* — LightGBM sees it alongside paid-session data and can
   learn the paid-session → physical gap on SFpark-covered blocks.
2. *Better off-hours target* — where SFpark coverage exists, we replace the
   heuristic diurnal formula with actual sensor occupancy for that hour.

**Permits** — an active blocking permit (excavation / emergency / temporary
occupancy) means the block is physically unavailable. Paid-meter occupancy
reads near zero on those blocks, which would teach the model "spaces are
free here" when the truth is "the block is closed". We (a) expose
`permit_active` as a feature, and (b) override the occupancy target to 95%
on permit-active rows so the model learns the correct direction.

In [9]:
def merge_sfpark_features(df):
    """Attach SFpark block-level physical occupancy per
    (blockface_id, hour, is_weekend). Zero-filled where no coverage."""
    path = os.path.join(DATA_DIR, "block_sfpark_features.csv")
    if not os.path.exists(path):
        print("  block_sfpark_features.csv missing — skipping SFpark merge.")
        df["sfpark_block_occ"] = 0.0
        df["sfpark_block_coverage"] = 0
        return df

    print("  Merging SFpark physical-occupancy features...")
    f = pd.read_csv(path)
    df = df.merge(
        f,
        left_on=["blockface_id", "hour", "is_weekend"],
        right_on=["blockface_id", "hour_of_day", "is_weekend"],
        how="left",
    )
    df = df.drop(columns=["hour_of_day"], errors="ignore")
    df["sfpark_block_occ"] = df["sfpark_block_occ"].fillna(0.0)
    df["sfpark_block_coverage"] = df["sfpark_block_coverage"].fillna(0).astype(int)
    coverage = df["sfpark_block_coverage"].mean() * 100
    print(f"    SFpark coverage: {coverage:.1f}% of rows")
    return df


def merge_permits_features(df):
    """Attach per-(cnn, date) active street-use permit flags via nearest
    master block to each training (lat, lon). Zero where no signal exists."""
    path = os.path.join(DATA_DIR, "block_permit_features.csv")
    master_path = os.path.join(PROJECT_DIR, "app", "models", "master_blocks.parquet")
    if not os.path.exists(path) or not os.path.exists(master_path):
        print("  Permit features or master_blocks missing — skipping permit merge.")
        df["permit_active"] = 0
        df["permit_count_30d"] = 0
        return df

    print("  Merging street-use permit features...")
    from scipy.spatial import cKDTree
    f = pd.read_csv(path, parse_dates=["date"])
    f["date"] = f["date"].dt.date

    master = pd.read_parquet(master_path)[["cnn", "lat", "lon"]]

    # Map each unique training (lat, lon) → nearest master cnn (same trick
    # as merge_poi_features). Permits are keyed on cnn, not blockface_id.
    unique_pts = df[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
    tree = cKDTree(master[["lat", "lon"]].values)
    _, idx = tree.query(unique_pts[["lat", "lon"]].values, k=1)
    unique_pts["cnn"] = master.iloc[idx]["cnn"].values

    df = df.merge(unique_pts, on=["lat", "lon"], how="left")
    df["date_key"] = df["timestamp"].dt.date

    df = df.merge(
        f,
        left_on=["cnn", "date_key"],
        right_on=["cnn", "date"],
        how="left",
    )
    df["permit_active"] = df["permit_active"].fillna(0).astype(int)
    df["permit_count_30d"] = df["permit_count_30d"].fillna(0).astype(int)
    df = df.drop(columns=["date", "date_key", "cnn"], errors="ignore")

    pct_active = df["permit_active"].mean() * 100
    print(f"    Permit-active coverage: {pct_active:.3f}% of rows")
    return df

In [10]:
CURB_TRANSIT_COLS = ["rpp_parcels_200m", "muni_stops_200m",
                     "shuttle_stops_200m", "disabled_signs_50m"]

def merge_curb_transit_features(df):
    """Attach RPP-parcel density, MUNI/shuttle stop counts, and disabled-
    parking sign counts. All keyed on cnn via nearest master block to each
    training (lat, lon) — same mapping pattern as merge_poi_features."""
    path = os.path.join(DATA_DIR, "block_curb_transit_features.csv")
    master_path = os.path.join(PROJECT_DIR, "app", "models", "master_blocks.parquet")
    if not os.path.exists(path) or not os.path.exists(master_path):
        print("  Curb/transit features or master_blocks missing — skipping.")
        for c in CURB_TRANSIT_COLS:
            df[c] = 0
        return df

    print("  Merging curb & transit features...")
    from scipy.spatial import cKDTree
    f = pd.read_csv(path)
    master = pd.read_parquet(master_path)[["cnn", "lat", "lon"]]

    unique_pts = df[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
    tree = cKDTree(master[["lat", "lon"]].values)
    _, idx = tree.query(unique_pts[["lat", "lon"]].values, k=1)
    unique_pts["cnn"] = master.iloc[idx]["cnn"].values
    unique_pts = unique_pts.merge(f, on="cnn", how="left")

    for c in CURB_TRANSIT_COLS:
        if c not in unique_pts.columns:
            unique_pts[c] = 0
        unique_pts[c] = unique_pts[c].fillna(0).astype(int)

    df = df.merge(unique_pts[["lat", "lon"] + CURB_TRANSIT_COLS],
                  on=["lat", "lon"], how="left")
    for c in CURB_TRANSIT_COLS:
        df[c] = df[c].fillna(0).astype(int)
    return df

## Pipeline

In [11]:
print("=" * 60)
print("ParkCast SF — Preprocessing Real Data")
print("=" * 60)

trans, locs, weather, sweeping, closures = load_data()

ParkCast SF — Preprocessing Real Data
Loading raw datasets...


### 1. Occupancy

In [12]:
df_processed = calculate_occupancy(trans, locs)

Calculating hourly occupancy from transactions (minutes-paid)...


  Blockfaces with >= 4 active meters: 1,413
  Active blockfaces: 1413 | Hours: 9070 | Grid: 12,815,910 rows


### 2. Features

In [13]:
df_final = enrich_features(df_processed, weather, sweeping, closures)

Enriching with features (Weather, School, Holiday, Events)...


  Merging weather...


  Merging events (continuous intensity)...


### 3. Merge citation signal (available 24/7, unlike paid meters)

In [14]:
df_final = merge_citations(df_final)
df_final = merge_poi_features(df_final)
df_final = merge_311_features(df_final)
df_final = merge_sfpark_features(df_final)
df_final = merge_permits_features(df_final)
df_final = merge_curb_transit_features(df_final)

# Flag whether meter enforcement is active. SFMTA standard is Mon-Sat
# 8am-6pm on most downtown/commercial blocks (not M-F 9-5). The
# previous formula threw away ~15-20% of real on-hours data, which was
# then overwritten by the synthetic off-hours formula.
df_final['is_meter_hours'] = (
    (df_final['day_of_week'] < 6)       # Mon-Sat
    & (df_final['hour'].between(8, 17)) # 8:00-17:59 = 8am-6pm
    & (df_final['is_holiday'] == 0)
).astype(int)

  Merging citation signal...


  Merging POI density features...


    POI coverage: 99.8% of rows
  Merging 311 complaint features...


    311 coverage: 6.2% of rows have nonzero median
  Merging SFpark physical-occupancy features...


    SFpark coverage: 13.3% of rows
  Merging street-use permit features...


    Permit-active coverage: 3.836% of rows
  Merging curb & transit features...


### 4. Off-hours target. Paid-meter occupancy is ~0 when meters are free,

In [15]:
#    which would teach the model to predict zero every night. No public
#    SF dataset measures 24/7 curb occupancy, so off-hour values are an
#    estimate anchored to real signals: each block's own daytime baseline,
#    a residential-style diurnal curve, the block's historical citation
#    demand at that hour, and any nearby event. Rows with estimated
#    targets are flagged via `target_is_estimated` so downstream work can
#    weight, filter, or report metrics separately.
baseline = (df_final[df_final['is_meter_hours'] == 1]
            .groupby(['lat', 'lon'])['occupancy_pct'].mean()
            .rename('blockface_baseline').reset_index())
df_final = df_final.merge(baseline, on=['lat', 'lon'], how='left')
df_final['blockface_baseline'] = df_final['blockface_baseline'].fillna(0.0)

# Diurnal multiplier for off-hours. Residential curb parking is typically
# fullest overnight (residents home) and emptiest mid-morning (commuters
# gone, meters just started). Values are relative to the daytime baseline.
DIURNAL = {
    0: 1.55, 1: 1.60, 2: 1.60, 3: 1.60, 4: 1.55, 5: 1.45,
    6: 1.25, 7: 1.05, 8: 0.95,
    18: 1.15, 19: 1.30, 20: 1.40, 21: 1.45, 22: 1.50, 23: 1.55,
}
diurnal_mult = df_final['hour'].map(DIURNAL).fillna(1.0).astype(float)

synthetic = (
    df_final['blockface_baseline'] * diurnal_mult
    + np.minimum(25.0, 12.0 * df_final['citations_hourly_median'])
    + np.minimum(15.0, 8.0 * df_final['complaints_311_median'])
    + 15.0 * df_final['event_intensity']
).clip(lower=0, upper=100)

off_mask = df_final['is_meter_hours'] == 0
df_final['target_is_estimated'] = off_mask.astype(int)
df_final.loc[off_mask, 'occupancy_pct'] = synthetic[off_mask].round(2)

# Where SFpark has actual physical-occupancy sensor data for this
# (block, hour, is_weekend), prefer it over the synthetic formula on
# off-hours. SFpark is 2011-13, but curb demand patterns (residents
# parking overnight, workday emptying) are slow-moving. Anchoring
# off-hours to sensor data on ~350 downtown blocks is strictly better
# than the hand-tuned DIURNAL curve.
sfpark_off = off_mask & (df_final['sfpark_block_coverage'] == 1)
df_final.loc[sfpark_off, 'occupancy_pct'] = (
    df_final.loc[sfpark_off, 'sfpark_block_occ'].round(2)
)
df_final.loc[sfpark_off, 'target_is_estimated'] = 0
print(f"  SFpark-anchored off-hours rows: {sfpark_off.sum():,}")

# A block under an active construction / emergency / temporary-occupancy
# permit is physically unavailable. Paid-meter occupancy reads ~0 on
# those rows (no one can pay a meter at a closed block), but the truth
# is "no spaces available". Override the target so the model learns
# permit_active=1 → predict high occupancy (= unavailable).
permit_mask = df_final['permit_active'] == 1
df_final.loc[permit_mask, 'occupancy_pct'] = 95.0
print(f"  Permit-override rows: {permit_mask.sum():,}")

print(f"  Off-hours target: {off_mask.sum():,} rows estimated. "
      f"Off-hours mean occ now {df_final.loc[off_mask, 'occupancy_pct'].mean():.2f}")
print("  By hour (off-hours only):")
print(df_final[off_mask].groupby('hour')['occupancy_pct'].mean().round(2).to_string())

final_cols = [
    'timestamp',
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday',
    'is_school_day', 'is_raining', 'temperature', 'event_intensity',
    'is_meter_hours', 'target_is_estimated',
    'citation_count', 'citations_hourly_median',
    'complaints_311_median', 'complaints_311_total',
    'poi_dining_200m', 'poi_retail_200m',
    'poi_transit_200m', 'poi_attraction_200m',
    'sfpark_block_occ', 'sfpark_block_coverage',
    'permit_active', 'permit_count_30d',
    'rpp_parcels_200m', 'muni_stops_200m',
    'shuttle_stops_200m', 'disabled_signs_50m',
    'neighborhood', 'lat', 'lon', 'total_spaces', 'occupancy_pct'
]
df_final = df_final[final_cols].fillna(0)

print(f"\nFinal dataset shape: {df_final.shape}")
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"SUCCESS: Saved processed data to {OUTPUT_PATH}")
print("=" * 60)

  SFpark-anchored off-hours rows: 1,101,680
  Permit-override rows: 491,640
  Off-hours target: 8,280,180 rows estimated. Off-hours mean occ now 44.05
  By hour (off-hours only):


hour
0     46.59
1     46.76
2     46.65
3     45.80
4     44.67
5     42.42
6     39.72
7     36.29
8     32.94
9     35.48
10    37.36
11    38.17
12    38.95
13    38.94
14    38.30
15    38.18
16    37.98
17    37.89
18    42.48
19    46.09
20    47.53
21    47.39
22    47.11
23    47.24



Final dataset shape: (12815910, 33)


SUCCESS: Saved processed data to /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/data/processed_training_data.csv
